In [1]:
%pip install cohere

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os
import numpy as np

In [3]:
load_dotenv()
cohereAPIKey=os.getenv("CohereAPIKey")

import cohere
co = cohere.Client(api_key=cohereAPIKey) # This is your trial API key

In [4]:
response = co.embed(
  model='embed-v4.0',
  texts=["Veena is a software engineer."],
  input_type='classification',
  truncate='NONE'
)
#print(f"Length of embedding: {len(response.embeddings)}")
print(f"Length of embedding: {len(response.embeddings[0])}")
print(f"Ebbeddings of text provided:\n{response.embeddings[0][:10]}")


Length of embedding: 1536
Ebbeddings of text provided:
[0.010484219, -0.011966281, 0.015698882, 0.020419525, 0.016796706, -0.020639092, -0.027555382, -0.029202117, 0.008892374, -0.0065869438]


In [49]:
def getEmbeddings(text:list,verbose) -> str:
    model="embed-v4.0"

    response = co.embed(
        model=model,
        texts=text,
        input_type="classification",
        truncate=None
    )

    return response.embeddings


In [45]:
embeddings = getEmbeddings(["Machine learning helps computers learn from data."],True)
print(f"first 10 Embeddings: {len(embeddings)}")

first 10 Embeddings: 1536


### Checking the cosine similarity.

In [7]:
def getCosineSimilarity(vec1,vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    cosineSimilarity = np.dot(vec1,vec2)/(np.linalg.norm(vec1) * np.linalg.norm(vec1))
    return cosineSimilarity 


In [8]:
textSample1 =[
    "I love Dhoni",
    "Dhoni's favourite shot is helicoptor shot",
    "I want to become an AI engineer",
    "I find FDE role interesting",
    "I love chicken biriyani",
    "Today, I ate mutton legs"
]

textSample2=[
    "I love playing cricket",
    "cricket is my favorite sport",
    "The stock market crashed today",
    "I enjoy watching football matches",
    "Python is a programming language"
]

embeddings = getEmbeddings(textSample2,False)
embDict={}

for i in range(len(textSample2)):
    embDict[textSample2[i]] = embeddings[i]


print(f"size of embDict: {len(embDict)}")


size of embDict: 5


In [9]:
keys=list(embDict.keys())
for i in range(len(keys)):
    for j in range(i+1,len(keys)):
        emb1 = embDict[keys[i]]
        emb2 = embDict[keys[j]]
        #print(emb1)
        #print(emb1)

        cosineSimilarity=getCosineSimilarity(emb1,emb2)
        print(f"Cosine similarity between '{keys[i]}' and '{keys[j]}' ::::: {cosineSimilarity}")
        


Cosine similarity between 'I love playing cricket' and 'cricket is my favorite sport' ::::: 0.8050736579840012
Cosine similarity between 'I love playing cricket' and 'The stock market crashed today' ::::: 0.3789348743985011
Cosine similarity between 'I love playing cricket' and 'I enjoy watching football matches' ::::: 0.6340784166211724
Cosine similarity between 'I love playing cricket' and 'Python is a programming language' ::::: 0.421246254132579
Cosine similarity between 'cricket is my favorite sport' and 'The stock market crashed today' ::::: 0.36687042961853744
Cosine similarity between 'cricket is my favorite sport' and 'I enjoy watching football matches' ::::: 0.5788733608130178
Cosine similarity between 'cricket is my favorite sport' and 'Python is a programming language' ::::: 0.39628249237709046
Cosine similarity between 'The stock market crashed today' and 'I enjoy watching football matches' ::::: 0.35215652482398596
Cosine similarity between 'The stock market crashed today

#### But everytime, we cannot take this cosine similarity as the only metric to check the similarity between two vectors. Because it does not support factual similarity. For example, if we have two vectors, one representing the word "Earth is round" and the other representing the word "Earth is flat", they may have a high cosine similarity because they are both animals, but they are not factually similar.

### Document search and retrieval

In [52]:
knowledgeBase = [
    "Machine learning helps computers learn from data.",
    "Deep learning uses neural networks with many layers.",
    "Artificial intelligence enables machines to perform human-like tasks.",
    "Python is a popular programming language for data science.",
    "Java is widely used for enterprise application development.",
    "Spring Boot simplifies Java backend development.",
    "FastAPI is a lightweight Python framework for building APIs.",
    "TensorFlow is a framework for training deep learning models.",
    "PyTorch is commonly used for AI research and experimentation.",
    "Natural language processing focuses on understanding human language.",
    "Vector databases store embeddings for efficient similarity search.",
    "ChromaDB is an open-source vector database.",
    "Redis can also be used as a vector database.",
    "Logistic regression is a classification algorithm.",
    "Random forests combine multiple decision trees.",
    "Support vector machines are supervised learning algorithms.",
    "Kubernetes automates container deployment and scaling.",
    "Docker packages applications into portable containers.",
    "Git is a distributed version control system.",
    "GitHub hosts Git repositories online."
]

#Embed all the statements.
embeddingsKB = getEmbeddings(knowledgeBase,False)
print(f"Total no.of sentences:{len(knowledgeBase)}, Total no.of embeddings: {len(embeddingsKB)}")
embeddingsDictKB={}
for i in range(len(knowledgeBase)):
    embeddingsDictKB[knowledgeBase[i]] = embeddingsKB[i]

print(f"length of embeddingsDictKB:{len(embeddingsDictKB)}")





Total no.of sentences:20, Total no.of embeddings: 20
length of embeddingsDictKB:20


### Function to find the top 3 similar sentences to the user query

In [77]:
def search(userQuery,embeddingKBLst, topK=3):
    userQryEmbedding = getEmbeddings(userQuery,False)
    print(f"length of User query embedding:{len(userQryEmbedding[0])}")

    scores=[]

    for i in range(len(embeddingKBLst)):
        #print(f"length of value: {len(value)}")
        score = getCosineSimilarity(userQryEmbedding[0],embeddingKBLst[i])
        scores.append((score,i))
    scores.sort(reverse=True)

    for rank, (score,indx) in enumerate(scores[:topK],1):
        print(f"{rank} - {score:.4f} ::: {knowledgeBase[indx]}")
    print()    

        

    

In [78]:
search(["How can computers understand human language?"],embeddingsKB,3)

length of User query embedding:1536
1 - 0.6117 ::: Natural language processing focuses on understanding human language.
2 - 0.4480 ::: Artificial intelligence enables machines to perform human-like tasks.
3 - 0.4473 ::: Machine learning helps computers learn from data.



In [79]:
search(["Which framework should I use to create a REST API in Python?"],embeddingsKB,3)

length of User query embedding:1536
1 - 0.6677 ::: FastAPI is a lightweight Python framework for building APIs.
2 - 0.4224 ::: Spring Boot simplifies Java backend development.
3 - 0.3949 ::: Python is a popular programming language for data science.



In [80]:
search(["Which algorithms can be used for classification?"],embeddingsKB,3)

length of User query embedding:1536
1 - 0.5483 ::: Logistic regression is a classification algorithm.
2 - 0.4998 ::: Support vector machines are supervised learning algorithms.
3 - 0.4280 ::: Python is a popular programming language for data science.



### Document Chunking
Curently, we saw, that we can generate embeddings for a single sentence. But in real world, we have documents which are very long and we cannot generate embeddings for the entire document. So, we need to chunk the document into smaller parts and then generate embeddings for each chunk. This is called document chunking.

### Vector Database
Now, we have generated embeddings for each chunk of the document. But how do we store these embeddings and retrieve them later? This is where vector databases come into play. A vector database is a database that is optimized for storing and retrieving high-dimensional vectors. Some popular vector databases are FAISS, Milvus, and Weaviate. Why not normal databases? Because normal databases are not optimized for high-dimensional vectors and they are not efficient in retrieving similar vectors, and it also leads to high latency. Eg: ChromaDB, Pinecode etc.

## CHUNKING

In [81]:
document = '''
Technology has transformed the way people communicate, work, and learn. From smartphones to cloud computing, digital innovations have become an essential part of everyday life. Organizations continue to adopt new technologies to improve efficiency, reduce costs, and provide better services to customers.

Education has also evolved significantly over the past decade. Online learning platforms allow students to access courses from anywhere in the world. Interactive videos, virtual classrooms, and digital assessments have made learning more flexible and accessible than ever before.

Healthcare is another field that has benefited from technological advancements. Electronic medical records, telemedicine, and wearable health devices enable doctors to monitor patients more effectively and provide timely medical care. Artificial intelligence is also being used to assist in disease diagnosis and medical research.

Environmental sustainability has become a global priority. Governments, businesses, and individuals are investing in renewable energy sources such as solar and wind power. Recycling, reducing waste, and conserving natural resources are important steps toward protecting the environment for future generations.

Transportation has improved with the development of electric vehicles, high-speed rail networks, and smart traffic management systems. These innovations aim to reduce pollution, improve safety, and make travel more efficient for millions of people around the world.

Businesses rely heavily on data to make informed decisions. Data analysis helps organizations identify trends, understand customer behavior, and optimize operations. Companies that effectively use data often gain a competitive advantage in their respective industries.

Cybersecurity plays a crucial role in protecting digital systems and sensitive information. Strong passwords, multi-factor authentication, encryption, and regular software updates help prevent unauthorized access and cyber attacks. Organizations also invest in security awareness training for employees.

Remote work has become increasingly common in many industries. Collaboration tools, video conferencing platforms, and cloud-based applications allow employees to work efficiently from different locations while staying connected with their teams.

Scientific research continues to drive innovation across various disciplines. Researchers conduct experiments, analyze results, and publish their findings to advance knowledge. Collaboration between universities, industries, and governments often leads to groundbreaking discoveries.

Personal development is an ongoing process of learning and self-improvement. Reading books, acquiring new skills, maintaining healthy habits, and setting meaningful goals help individuals grow both personally and professionally throughout their lives.
'''

print(f"length of Document: {len(document)} characters")
print(f"length of Document: {len(document.split())} words")

length of Document: 2857 characters
length of Document: 369 words


In [85]:
def chunkBypara(data,verbose):
    '''Split chunks at the end of every paragraph, so that each paragraph is a chunk.'''

    chunks = [para.strip() for para in data.strip().split("\n\n")]


    if verbose:
        print(f"chunks: {chunks}")
    return chunks

In [101]:
chunks = chunkBypara(document,True)
print(f"No.of chunks created: {len(chunks)}")

chunks: ['Technology has transformed the way people communicate, work, and learn. From smartphones to cloud computing, digital innovations have become an essential part of everyday life. Organizations continue to adopt new technologies to improve efficiency, reduce costs, and provide better services to customers.', 'Education has also evolved significantly over the past decade. Online learning platforms allow students to access courses from anywhere in the world. Interactive videos, virtual classrooms, and digital assessments have made learning more flexible and accessible than ever before.', 'Healthcare is another field that has benefited from technological advancements. Electronic medical records, telemedicine, and wearable health devices enable doctors to monitor patients more effectively and provide timely medical care. Artificial intelligence is also being used to assist in disease diagnosis and medical research.', 'Environmental sustainability has become a global priority. Govern

## Initialise Vector DB

In [91]:
%pip install chromadb

  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl (23.5 MB)



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [93]:
import chromadb

In [95]:
chromaDB = chromadb.Client()

### Creating collection in Chroma DB

In [99]:
collection = chromaDB.create_collection(
    name="PedhaBalaSiksha",
    metadata={
        "description":"Info about all matters in the world."
    }
)

InternalError: Collection [PedhaBalaSiksha] already exists

### Adding chunks to DB

In [100]:
collection.add(
    documents=chunks,
    ids=[f"chunk_{id}" for id in range(len(chunks))]
)

C:\Users\HP\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:47<00:00, 1.77MiB/s]


- From above, we can see something called as "all-MiniLM-L6-v2". This is an internal default model used by ChromaDB to generate embeddings for the chunks. But we can also use our own embeddings model to generate embeddings for the chunks and then store them in ChromaDB.
- We can also externally, generate the embeddings and then store them in ChromaDB. Eg:
`
collection.add(
    ids=["1", "2"],
    documents=chunks,
    embeddings=embeddings
)
`

### Querying the vector database

In [107]:
results = collection.query(
    query_texts=["Technology"], # Chroma will embed this for you
    n_results=1 # how many results to return
)
print(results)

{'ids': [['chunk_0']], 'embeddings': None, 'documents': [['Technology has transformed the way people communicate, work, and learn. From smartphones to cloud computing, digital innovations have become an essential part of everyday life. Organizations continue to adopt new technologies to improve efficiency, reduce costs, and provide better services to customers.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None]], 'distances': [[0.9016420841217041]]}
